# Proyecto 01: Retail Intelligence
## A2: Análisis de Rentabilidad y Comportamiento de Compra
**Asignatura:** Tópicos Especiales para la Gestión de Datos  
**Caso:** RetailVenture C.A.  
**Autor:** Daniel Azuaje (Equipo RetailVenture)

---

### Objetivos de la Actividad:
1. **A2.1 Contribución al Margen Bruto:** Identificar segmentos, categorías y canales con mayor/menor aporte al margen total.
2. **A2.2 Customer Lifetime Value (CLV):** Estimar el CLV por segmento e identificar segmentos en declive.
3. **A2.3 Estacionalidad:** Analizar picos, valles y tendencias temporales por categoría y canal (2023 vs 2024).
4. **A2.4 Efecto de Descuentos:** Evaluar si el volumen compensa la pérdida de margen y detectar ventas bajo costo.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo visual ejecutivo
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Fuente de datos: lectura local de clean (o conexión a Supabase)
DATA_PATH = Path("../data/clean")

df_trx = pd.read_csv(DATA_PATH / "transacciones_maestra.csv", parse_dates=["fecha"])
df_cli = pd.read_csv(DATA_PATH / "clientes_maestra.csv")
df_prd = pd.read_csv(DATA_PATH / "productos_limpio.csv")

print(f"Transacciones cargadas: {len(df_trx):,}")
print(f"Ingreso neto total:  ${df_trx['monto_neto'].sum():,.2f}")
print(f"Margen bruto total:  ${df_trx['margen_bruto'].sum():,.2f} ({100*df_trx['margen_bruto'].sum()/df_trx['monto_neto'].sum():.2f}% del ingreso)")


---
## A2.1 Contribución al Margen Bruto

Analizamos la rentabilidad multidimensional:
- **Por Categoría de Producto**
- **Por Canal de Venta**
- **Por Segmento de Cliente**
- **Cruce Causal:** Canal × Categoría (¿Dónde se destruye margen?)


In [ ]:
# Función para resumen de contribución
TOTAL_INGRESO = df_trx['monto_neto'].sum()
TOTAL_MARGEN = df_trx['margen_bruto'].sum()

def tabla_contribucion(df, dimension):
    res = df.groupby(dimension).agg(
        transacciones=('transaccion_id', 'count'),
        ingreso_neto=('monto_neto', 'sum'),
        margen_bruto=('margen_bruto', 'sum'),
        margen_neto_devol=('margen_neto_devol', 'sum'),
        tasa_devolucion=('devuelto', 'mean'),
        descuento_promedio=('descuento', 'mean')
    ).reset_index()
    
    res['pct_ingreso'] = (100 * res['ingreso_neto'] / TOTAL_INGRESO).round(2)
    res['pct_margen'] = (100 * res['margen_bruto'] / TOTAL_MARGEN).round(2)
    res['margen_pct_real'] = (100 * res['margen_bruto'] / res['ingreso_neto']).round(2)
    res['tasa_devolucion'] = (100 * res['tasa_devolucion']).round(2)
    res['descuento_promedio'] = (100 * res['descuento_promedio']).round(2)
    
    return res.sort_values(by='margen_bruto', ascending=False)


### 1. Rentabilidad por Categoría de Producto

In [ ]:
cat_summary = tabla_contribucion(df_trx, 'categoria')
cat_summary[['categoria', 'ingreso_neto', 'pct_ingreso', 'margen_bruto', 'pct_margen', 'margen_pct_real', 'tasa_devolucion']]


In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))

# Gráfico dual: Volumen de Ingreso vs Margen Bruto
x = np.arange(len(cat_summary))
width = 0.35

rects1 = ax1.bar(x - width/2, cat_summary['pct_ingreso'], width, label='% Ventas (Ingreso)', color='#4A90E2')
rects2 = ax1.bar(x + width/2, cat_summary['pct_margen'], width, label='% Margen Bruto Aportado', color='#50E3C2')

ax1.set_ylabel('% del Total de la Empresa')
ax1.set_title('Aporte al Ingreso vs. Aporte al Margen por Categoría', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(cat_summary['categoria'], rotation=25, ha='right')
ax1.legend()
plt.tight_layout()
plt.show()


### 2. Rentabilidad por Canal de Venta

In [ ]:
canal_summary = tabla_contribucion(df_trx, 'canal')
canal_summary[['canal', 'ingreso_neto', 'pct_ingreso', 'margen_bruto', 'pct_margen', 'margen_pct_real', 'descuento_promedio']]


In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=canal_summary, x='canal', y='margen_pct_real', palette='Blues_d')
plt.title('Margen Bruto Real por Canal (%)', fontsize=12, fontweight='bold')
plt.ylabel('Margen Bruto %')
plt.xlabel('Canal')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 4),
                ha='center', va='center', color='white', fontweight='bold')
plt.show()


### 3. Rentabilidad por Segmento de Clientes (Pareto)

In [ ]:
seg_summary = tabla_contribucion(df_trx, 'segmento')
seg_summary[['segmento', 'ingreso_neto', 'pct_ingreso', 'margen_bruto', 'pct_margen', 'margen_pct_real']]


### 4. Cruce Causal: Descuento Online vs. Destrucción de Margen

In [ ]:
cruce_desc = df_trx.pivot_table(index='categoria', columns='canal', values='descuento', aggfunc=lambda s: (s.mean()*100).round(2))
cruce_margen = df_trx.pivot_table(index='categoria', columns='canal', values='margen_pct_real', aggfunc='mean').round(2)

print("--- Descuento Promedio (%) por Categoría y Canal ---")
display(cruce_desc)

print("\n--- Margen Real (%) por Categoría y Canal ---")
display(cruce_margen)


---
## A2.2 Customer Lifetime Value (CLV) por Segmento
*(Próximo paso: calibrar con la tasa de churn del equipo)*


In [ ]:
# Esqueleto preliminar de CLV
# CLV = Margen promedio por compra x Compras anuales x Vida esperada (1 / churn)
# En desarrollo...


---
## A2.3 Análisis de Estacionalidad de Ventas (2023 vs 2024)
*(Próximo paso: descomposición mensual y picos/valles)*


In [ ]:
# En desarrollo...


---
## A2.4 Elasticidad y Efecto de Descuentos
*(Próximo paso: análisis de punto de equilibrio y ventas bajo costo)*


In [ ]:
# En desarrollo...
